# Prepare Datasets for Machine Learning

In previous notebooks, SNOTEL, PRISM, and DEM datasets were downloaded and cleaned in preparation for application to machine learning. This notebook will perform final cleaning and calculate some further statistics for the PRISM dataset; the following notebook will perform the machine learning.

## Step 1: Import Libraries and Set Up Project Directory

In [1]:
# import libraries

# file management
import os
import pathlib
from pathlib import Path
import sys

# datatypes
import numpy as np
import pandas as pd
import xarray as xr

# geospatial data
import geopandas as gpd
import rioxarray as rxr

# plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [2]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

## Step 2: Station Filtering

This step will filter the SNOTEL stations used in the ML model to stations with at least 20 years of record keeping. Ideally, stations with 30 years would be used. However, this will likely disqualify too many stations and reduce the dataset size too much.

#### Step 2a: Import PRISM

In [3]:
# Import PRISM dataset

# set a path
prism_path = Path(cleaned_data_dir, 'prism', 'prism_mhw_1990_2020_cleaned_with_crs.nc')

# open prism dataset
prism_ds = xr.open_dataset(prism_path, decode_coords='all')

# check on DS
print(prism_ds.rio.crs)
prism_ds

EPSG:4326


<xarray.Dataset> Size: 372MB
Dimensions:  (time: 7328, lat: 51, lon: 83)
Coordinates:
  * lon      (lon) float64 664B -113.9 -113.9 -113.8 ... -110.6 -110.5 -110.5
  * lat      (lat) float64 408B 46.46 46.42 46.38 46.33 ... 44.46 44.42 44.38
  * time     (time) datetime64[ns] 59kB 1990-10-01 1990-10-02 ... 2020-06-01
    crs      int64 8B ...
Data variables:
    ppt      (time, lat, lon) float32 124MB ...
    tmin     (time, lat, lon) float32 124MB ...
    tmax     (time, lat, lon) float32 124MB ...
Attributes:
    Conventions:  CF-1.5
    GDAL:         GDAL 3.12.0 "Chicoutimi", released 2025/11/03
    history:      Thu Jun 18 17:05:52 2026: GDAL CreateCopy( /nfs/pancake/u5/...

#### Step 2b: Import SNOTEL

In [4]:
# Import SNOTEL dataset

# set a path
snotel_path = Path(cleaned_data_dir, 'bcqc_snotel_1990-2020_RAW.nc')

# open SNOTEL dataset
snotel_ds = xr.open_dataset(snotel_path, decode_coords='all')

# check ds
snotel_ds

<xarray.Dataset> Size: 11MB
Dimensions:          (stationTriplet: 26, date: 10837)
Coordinates:
  * date             (date) datetime64[ns] 87kB 1990-10-01 ... 2020-06-01
  * stationTriplet   (stationTriplet) <U11 1kB '916:MT:SNTL' ... '384:WY:SNTL'
    stationId        (stationTriplet) <U3 312B ...
    name             (stationTriplet) <U16 2kB ...
    latitude         (stationTriplet) float64 208B ...
    longitude        (stationTriplet) float64 208B ...
    beginDate        (stationTriplet) <U16 2kB ...
    endDate          (stationTriplet) <U16 2kB ...
Data variables:
    daily_precip_in  (stationTriplet, date) float64 2MB ...
    tmax_f           (stationTriplet, date) float64 2MB ...
    tmin_f           (stationTriplet, date) float64 2MB ...
    tavg_f           (stationTriplet, date) float64 2MB ...
    SWE              (stationTriplet, date) float64 2MB ...

In [6]:
# check units

print(snotel_ds.isel(stationTriplet=slice(0, 2), date=slice(70, 73)).to_dataframe())

                           daily_precip_in  tmax_f  tmin_f  tavg_f  SWE  \
stationTriplet date                                                       
916:MT:SNTL    1990-12-10              NaN     NaN     NaN     NaN  NaN   
               1990-12-11              NaN     NaN     NaN     NaN  NaN   
               1990-12-12              NaN     NaN     NaN     NaN  NaN   
318:MT:SNTL    1990-12-10              0.0   38.62   28.32   32.44  1.6   
               1990-12-11              0.1   30.38   -0.52   19.05  1.6   
               1990-12-12              0.1   13.90   -4.64    1.54  1.6   

                          stationId            name  latitude  longitude  \
stationTriplet date                                                        
916:MT:SNTL    1990-12-10       916      Albro Lake  45.59723 -111.95902   
               1990-12-11       916      Albro Lake  45.59723 -111.95902   
               1990-12-12       916      Albro Lake  45.59723 -111.95902   
318:MT:SNTL    1990

Units are all imperial. The next code will add data variables that have been converted to metric.

In [7]:
# Unit Conversion

# set conversion factor
inch_to_mm = 25.4

# Perform conversions
snotel_ds = snotel_ds.assign(
    # Temp conversion
    tavg_c = (snotel_ds['tavg_f'] - 32) * (5/9),
    tmin_c = (snotel_ds['tmin_f'] - 32) * (5/9),
    tmax_c = (snotel_ds['tmax_f'] - 32) * (5/9),

    # Precip conversion
    daily_precip_mm = snotel_ds['daily_precip_in'] * inch_to_mm,
    swe_mm = snotel_ds['SWE'] * inch_to_mm
)

#### Step 2c: Filter PRISM at SNOTEL sites

In [8]:
# extract prism values at station locations

# initialize list
prism_filtered_list = []

# get length of stations
num_stations = len(snotel_ds.stationTriplet)

# filter through prism and extract timeseries for each pixel that has a snotel station in it
for i in range(num_stations):
    st_id = snotel_ds['stationTriplet'].values[i]
    lat = float(snotel_ds['latitude'].values[i])
    lon = float(snotel_ds['longitude'].values[i])

    # filter prism dataset
    prism_pixel = prism_ds.sel(
        # grab pixel that matches station coords
        lon = lon, lat = lat,
        # get the pixel nearest to the station
        method='nearest'
        # convert to DF
        ).to_dataframe().reset_index()
    
    # keep only climate and time data
    prism_pixel_df = prism_pixel[
        # grab just time, ppt, tmin, tmax
        ['time', 'ppt', 'tmin', 'tmax']
        # rename columns to remind that it's prism data
        ].rename(
            columns = {
                'time': 'date',
                'ppt': 'prism_ppt_mm',
                'tmin': 'prism_tmin_c',
                'tmax': 'prism_tmax_c'
            }
        )
    
    # append station Triplet to it can be joined to the SNOTEL dataset
    prism_pixel_df['stationTriplet'] = st_id

    # append to list
    prism_filtered_list.append(prism_pixel_df)

# concat prism dfs
prism_filter_df = pd.concat(prism_filtered_list, ignore_index=True)

In [9]:
# check the df
prism_filter_df

,date,prism_ppt_mm,prism_tmin_c,prism_tmax_c,stationTriplet
0,1990-10-01,0.000,0.324,14.996000,916:MT:SNTL
1,1990-10-02,0.000,2.630,15.985000,916:MT:SNTL
2,1990-10-03,1.948,-6.244,4.975000,916:MT:SNTL
3,1990-10-04,0.000,-6.558,2.983000,916:MT:SNTL
4,1990-10-05,1.476,2.839,12.372000,916:MT:SNTL
...,...,...,...,...,...
190523,2020-05-28,0.000,0.379,17.624001,384:WY:SNTL
190524,2020-05-29,0.000,0.574,20.450001,384:WY:SNTL
190525,2020-05-30,0.000,1.982,22.357000,384:WY:SNTL
190526,2020-05-31,0.072,3.837,24.507000,384:WY:SNTL


#### Step 2d: Count how many years of data each SNOTEL station has

In [10]:
# check start years

# get start year values
start_years = snotel_ds.beginDate.values

# convert to datetime
start_years_dt = pd.to_datetime(start_years)

# check how many stations started by 1990 and by 1996
# that's 30 or 25 years of data, respectively
print(sum(start_years_dt <= '1990-10-01'))
print(sum(start_years_dt <= '1996-10-01'))

# check overall number of stations
print(len(start_years))

23
25
26


Starting by 1990 keeps 23 stations, while starting five years later keeps 25 stations. One station is lost either way. Preserving spatial resolution seems more important than preserving temporal resolution, so I'll filter to stations starting by 1996.

In [11]:
# set mask based on station start date
station_mask = start_years_dt <= pd.to_datetime('1996-10-01')

# filter ds
snotel_filter_ds = snotel_ds.sel(stationTriplet = station_mask)

# check that filtering worked
print(f"snotel_ds has {len(snotel_ds.stationTriplet)} stations")
print(f"snotel_filter_ds has {len(snotel_filter_ds.stationTriplet)} stations")

snotel_ds has 26 stations
snotel_filter_ds has 25 stations


#### Step 2e: Convert SNOTEL to DataFrame

In [12]:
# convert from DS to DF
snotel_full_df = snotel_filter_ds.to_dataframe().reset_index()

# remove 1990-1995 years (the previous step didn't filter temporally)
# set date to datetime
snotel_full_df['date'] = pd.to_datetime(snotel_full_df['date'])

# set start and end dates
start_date = '1996-10-01'
end_date = '2020-06-01'

# filter using start and end dates
snotel_time_filter_df = snotel_full_df[(snotel_full_df['date'] >= start_date) & (snotel_full_df['date'] <= end_date)]

# set columns to keep
# filter out imperial columns, and drop SNOTEL weather (tmin, tmax, daily precip)
columns_keep = [
    # metadata
    'stationTriplet', 'date', 'latitude', 'longitude',
    # data
    #'tavg_c', 'tmin_c', 'tmax_c', 'daily_precip_mm', 
    'swe_mm']

# filter
snotel_df = snotel_time_filter_df[columns_keep]

# check it out
snotel_df

,stationTriplet,date,latitude,longitude,swe_mm
2192,916:MT:SNTL,1996-10-01,45.59723,-111.95902,0.00
2193,916:MT:SNTL,1996-10-02,45.59723,-111.95902,0.00
2194,916:MT:SNTL,1996-10-03,45.59723,-111.95902,0.00
2195,916:MT:SNTL,1996-10-04,45.59723,-111.95902,0.00
2196,916:MT:SNTL,1996-10-05,45.59723,-111.95902,0.00
...,...,...,...,...,...
270920,384:WY:SNTL,2020-05-28,44.71961,-110.51084,10.16
270921,384:WY:SNTL,2020-05-29,44.71961,-110.51084,0.00
270922,384:WY:SNTL,2020-05-30,44.71961,-110.51084,0.00
270923,384:WY:SNTL,2020-05-31,44.71961,-110.51084,0.00


## Step 3: Geospatial Joins

This step will join the PRISM, SNOTEL, and DEM datasets into a single dataframe.

#### Step 3a: Join PRISM and SNOTEL datasets

In [13]:
# merge PRISM and SNOTEL datasets
prism_snotel_df = pd.merge(
    snotel_df,
    prism_filter_df,
    on = ['stationTriplet', 'date'],
    how = 'inner'
)

# check it out
prism_snotel_df

,stationTriplet,date,latitude,longitude,swe_mm,prism_ppt_mm,prism_tmin_c,prism_tmax_c
0,916:MT:SNTL,1996-10-01,45.59723,-111.95902,0.00,0.000,4.278,16.568001
1,916:MT:SNTL,1996-10-02,45.59723,-111.95902,0.00,0.001,-2.904,13.110000
2,916:MT:SNTL,1996-10-03,45.59723,-111.95902,0.00,0.000,-1.004,16.767000
3,916:MT:SNTL,1996-10-04,45.59723,-111.95902,0.00,0.000,2.815,16.663000
4,916:MT:SNTL,1996-10-05,45.59723,-111.95902,0.00,0.000,3.384,16.459999
...,...,...,...,...,...,...,...,...
146545,384:WY:SNTL,2020-05-28,44.71961,-110.51084,10.16,0.000,0.379,17.624001
146546,384:WY:SNTL,2020-05-29,44.71961,-110.51084,0.00,0.000,0.574,20.450001
146547,384:WY:SNTL,2020-05-30,44.71961,-110.51084,0.00,0.000,1.982,22.357000
146548,384:WY:SNTL,2020-05-31,44.71961,-110.51084,0.00,0.072,3.837,24.507000


#### Step 3b: Import DEM datasets

In [5]:
# import DEM datasets

# set topo dir
topo_rsmp_dir = Path(cleaned_data_dir, 'topo')

# paths
elev_4km_path = Path(topo_rsmp_dir, 'mhw_elevation_4km.tif')
slope_4km_path = Path(topo_rsmp_dir, 'mhw_slope_4km.tif')
aspect_4km_path = Path(topo_rsmp_dir, 'mhw_aspect_4km.tif')
aspect_north_4km_path = Path(topo_rsmp_dir, 'mhw_northness_4km.tif')
aspect_east_4km_path = Path(topo_rsmp_dir, 'mhw_eastness_4km.tif')

# import
mhw_elev_4km_da = rxr.open_rasterio(elev_4km_path)
mhw_slope_4km_da = rxr.open_rasterio(slope_4km_path)
mhw_aspect_4km_da = rxr.open_rasterio(aspect_4km_path)
mhw_aspect_4km_north_da = rxr.open_rasterio(aspect_north_4km_path)
mhw_aspect_4km_east_da = rxr.open_rasterio(aspect_east_4km_path)

# merge into one dataset
dem_dataset = xr.Dataset({
    'elevation': mhw_elev_4km_da,
    'slope': mhw_slope_4km_da,
    'aspect': mhw_aspect_4km_da,
    'aspect_north': mhw_aspect_4km_north_da,
    'aspect_east': mhw_aspect_4km_east_da
})

# make sure dataset is in ESPG: 4326
if dem_dataset.rio.crs.to_epsg() != 4326:
    dem_dataset = dem_dataset.rio.reproject("EPSG:4326")

In [6]:
# check it out
dem_dataset

<xarray.Dataset> Size: 77kB
Dimensions:       (band: 1, x: 83, y: 51)
Coordinates:
  * band          (band) int64 8B 1
  * x             (x) float64 664B -113.9 -113.9 -113.8 ... -110.6 -110.5 -110.5
  * y             (y) float64 408B 46.46 46.42 46.38 46.33 ... 44.46 44.42 44.38
    spatial_ref   int64 8B 0
Data variables:
    elevation     (band, y, x) int16 8kB ...
    slope         (band, y, x) float32 17kB ...
    aspect        (band, y, x) float32 17kB ...
    aspect_north  (band, y, x) float32 17kB ...
    aspect_east   (band, y, x) float32 17kB ...

#### Step 3c: Select DEM data at stations

In [15]:
# grab each station's point
stations_point_df = (prism_snotel_df[['stationTriplet', 'latitude', 'longitude']]
                     # get rid of duplicate lines, only need one row per station
                     .drop_duplicates().reset_index())

# grab coordinates for each station
x_coords = xr.DataArray(
    stations_point_df['longitude'], 
    dims="stationTriplet", 
    coords={"stationTriplet": stations_point_df['stationTriplet']})
y_coords = xr.DataArray(
    stations_point_df['latitude'], 
    dims="stationTriplet", 
    coords={"stationTriplet": stations_point_df['stationTriplet']})

# search through DEM dataset and grab all stats for each point
station_dem_ds = dem_dataset.sel(x = x_coords, y = y_coords, method = 'nearest')
station_dem_ds = station_dem_ds.squeeze('band', drop=True)

# convert to dataframe before merging
station_dem_df = station_dem_ds.to_dataframe().reset_index()

# drop extra columns
station_dem_df = station_dem_df[['stationTriplet', 'elevation', 'slope', 'aspect_north', 'aspect_east']]

# check it out
station_dem_df

,stationTriplet,elevation,slope,aspect_north,aspect_east
0,916:MT:SNTL,2546,20.723917,0.208108,0.978106
1,318:MT:SNTL,2996,21.693270,-0.455876,0.890043
2,328:MT:SNTL,2532,14.942879,-0.008403,-0.999965
3,347:MT:SNTL,2479,2.508581,0.843661,0.536875
4,355:MT:SNTL,2343,8.982809,-0.242535,-0.970143
5,365:MT:SNTL,2372,23.627708,-0.032591,-0.999469
6,381:MT:SNTL,1970,4.599943,-0.728200,0.685365
7,385:MT:SNTL,2722,9.332155,-0.977648,0.210247
8,403:MT:SNTL,2540,23.873613,0.955363,-0.295434
9,436:MT:SNTL,2718,18.510357,0.055470,0.998460


#### 3d: Join DEM with PRISM/SNOTEL df

In [16]:
# join on station triplet
stations_ml_df = pd.merge(
    prism_snotel_df,
    station_dem_df,
    on = 'stationTriplet',
    how = 'left'
)

# check it out
stations_ml_df

,stationTriplet,date,latitude,longitude,swe_mm,prism_ppt_mm,prism_tmin_c,prism_tmax_c,elevation,slope,aspect_north,aspect_east
0,916:MT:SNTL,1996-10-01,45.59723,-111.95902,0.00,0.000,4.278,16.568001,2546,20.723917,0.208108,0.978106
1,916:MT:SNTL,1996-10-02,45.59723,-111.95902,0.00,0.001,-2.904,13.110000,2546,20.723917,0.208108,0.978106
2,916:MT:SNTL,1996-10-03,45.59723,-111.95902,0.00,0.000,-1.004,16.767000,2546,20.723917,0.208108,0.978106
3,916:MT:SNTL,1996-10-04,45.59723,-111.95902,0.00,0.000,2.815,16.663000,2546,20.723917,0.208108,0.978106
4,916:MT:SNTL,1996-10-05,45.59723,-111.95902,0.00,0.000,3.384,16.459999,2546,20.723917,0.208108,0.978106
...,...,...,...,...,...,...,...,...,...,...,...,...
146545,384:WY:SNTL,2020-05-28,44.71961,-110.51084,10.16,0.000,0.379,17.624001,2387,7.114066,-0.404747,-0.914429
146546,384:WY:SNTL,2020-05-29,44.71961,-110.51084,0.00,0.000,0.574,20.450001,2387,7.114066,-0.404747,-0.914429
146547,384:WY:SNTL,2020-05-30,44.71961,-110.51084,0.00,0.000,1.982,22.357000,2387,7.114066,-0.404747,-0.914429
146548,384:WY:SNTL,2020-05-31,44.71961,-110.51084,0.00,0.072,3.837,24.507000,2387,7.114066,-0.404747,-0.914429


## Step 4: NaN Removal

This step will remove any remaining NaNs from the PRISM and SNOTEL datasets using interpolation or regression. 

I can't just use .dropna(), as I need to preserve the daily structure of the record. Thus, some type of interpolation is required. However, the methodology is important here, as temp and precip can't be interpolated in the same ways, given their inherent differences.



In [17]:
# check for NaNs
print(stations_ml_df.isna().sum())

stationTriplet        0
date                  0
latitude              0
longitude             0
swe_mm            15643
prism_ppt_mm          0
prism_tmin_c          0
prism_tmax_c          0
elevation             0
slope                 0
aspect_north          0
aspect_east           0
dtype: int64


In [18]:
# sort df by station triplet and date 
stations_ml_df = stations_ml_df.sort_values(by=['stationTriplet', 'date']).reset_index(drop=True)

In [19]:
# linear interpolation for SWE
stations_ml_df['swe_mm'] = stations_ml_df.groupby('stationTriplet')['swe_mm'].transform(
    lambda x: x.interpolate(method='linear'))

In [20]:
# check again for NaNs
print(stations_ml_df.isna().sum())

stationTriplet       0
date                 0
latitude             0
longitude            0
swe_mm            2929
prism_ppt_mm         0
prism_tmin_c         0
prism_tmax_c         0
elevation            0
slope                0
aspect_north         0
aspect_east          0
dtype: int64


Using interpolation can't patch all of the missing data, unfortunately. These last NaNs will be dropped at the very end of the prep process.

## Step 5: Statistics Calculations

This step will calculate additional statistics on the PRISM and SNOTEL stations, such as rolling averages of temperature and precipitation. [Moya et al. 2026](https://doi.org/10.5194/tc-20-1427-2026) found that lagged variables were impactful, but not by much after 3 days for air temp and certainly not helpful after 7 days. 

#### Step 5a: Calculate Statistics on SNOTEL/PRISM/DEM df

In [21]:
# set station df index to date so Pandas can calculate rolling windows
stations_ml_df = stations_ml_df.set_index('date')

# rolling 3 day window of mean PRISM tmin
stations_ml_df['prism_tmin_c_roll_3d'] = (stations_ml_df
                                          # group by each station and tmin
                                          .groupby('stationTriplet')['prism_tmin_c']
                                          # calculate rolling three day mean
                                          .transform(
                                              lambda x: x.rolling('3D').mean()
                                          ))

# rolling 3 day window of mean PRISM tmax
stations_ml_df['prism_tmax_c_roll_3d'] = (stations_ml_df
                                          # group by each station and tmin
                                          .groupby('stationTriplet')['prism_tmax_c']
                                          # calculate rolling three day mean
                                          .transform(
                                              lambda x: x.rolling('3D').mean()
                                          ))

# rolling 3 day window of PRISM precip sum
# sum is better than mean for precip, as precip is non-continuous
stations_ml_df['prism_ppt_mm_roll_3d'] = (stations_ml_df
                                          .groupby('stationTriplet')['prism_ppt_mm']
                                          .transform(
                                              lambda x: x.rolling('3D').sum()
                                          ))


In [22]:
# create 7 day windows; informed by Alabi et al. 2026

# rolling 7 day window of mean PRISM tmin
stations_ml_df['prism_tmin_c_roll_7d'] = (stations_ml_df
                                          # group by each station and tmin
                                          .groupby('stationTriplet')['prism_tmin_c']
                                          # calculate rolling three day mean
                                          .transform(
                                              lambda x: x.rolling('7D').mean()
                                          ))

# rolling 7 day window of mean PRISM tmax
stations_ml_df['prism_tmax_c_roll_7d'] = (stations_ml_df
                                          # group by each station and tmin
                                          .groupby('stationTriplet')['prism_tmax_c']
                                          # calculate rolling three day mean
                                          .transform(
                                              lambda x: x.rolling('7D').mean()
                                          ))

# rolling 7 day window of PRISM precip sum
# sum is better than mean for precip, as precip is non-continuous
stations_ml_df['prism_ppt_mm_roll_7d'] = (stations_ml_df
                                          .groupby('stationTriplet')['prism_ppt_mm']
                                          .transform(
                                              lambda x: x.rolling('7D').sum()
                                          ))

# reset index
stations_ml_df = stations_ml_df.reset_index()

In [23]:
# get cumulative precip per water year

# create a water year variable
shifted_year = stations_ml_df['date'].dt.year + (stations_ml_df['date'].dt.month >= 10).astype(int)
stations_ml_df['water_year'] = shifted_year

# create a tmean variable
stations_ml_df['prism_tmean_c'] = (stations_ml_df['prism_tmin_c'] + stations_ml_df['prism_tmax_c']) / 2

# get a cumulative precip variable
# this groups by station and water year
stations_ml_df['wy_cumul_precip_mm'] = (stations_ml_df
                                        .groupby(['stationTriplet', 'water_year'])['prism_ppt_mm']
                                        .cumsum())

# get a cumulative sum of degrees above freezing on days when tmean > freezing
# remove any values below freezing (0 deg C)
stations_ml_df['positive_tmean_days'] = stations_ml_df['prism_tmean_c'].clip(lower=0)
# then get cumulative sum
stations_ml_df['wy_cumul_melt_days'] = (
    stations_ml_df
    .groupby(['stationTriplet', 'water_year'])['prism_tmean_c']
    .cumsum()
)

# drop positive_tmean_days
stations_ml_df = stations_ml_df.drop(columns=['positive_tmean_days'])

# add day of water year (Alabi et al. 2026 highlight the importance of this feature)
# get water year start date
wy_start_date = pd.to_datetime(
    # get water year, subtract 1 (since the water year started the previous year)
    (stations_ml_df['water_year'] - 1)
    # set to a string and add the month and day start
    .astype(str) + '-10-01'
    )
# get day of water year
stations_ml_df['dowy'] = (
    # subtract the start date from each row's date
    (stations_ml_df['date'] - wy_start_date)
    # set to days and add 1 so that the DOWY starts at 1
    .dt.days + 1)

In [24]:
# check the df
stations_ml_df

,date,stationTriplet,latitude,longitude,swe_mm,prism_ppt_mm,prism_tmin_c,prism_tmax_c,elevation,slope,...,prism_tmax_c_roll_3d,prism_ppt_mm_roll_3d,prism_tmin_c_roll_7d,prism_tmax_c_roll_7d,prism_ppt_mm_roll_7d,water_year,prism_tmean_c,wy_cumul_precip_mm,wy_cumul_melt_days,dowy
0,1996-10-01,318:MT:SNTL,44.47147,-112.98191,NaN,0.000,2.544,16.346001,2996,21.693270,...,16.346001,0.000,2.544000,16.346001,0.000,1997,9.445001,0.000000,9.445001,1
1,1996-10-02,318:MT:SNTL,44.47147,-112.98191,NaN,0.000,-0.064,13.597000,2996,21.693270,...,14.971500,0.000,1.240000,14.971500,0.000,1997,6.766500,0.000000,16.211500,2
2,1996-10-03,318:MT:SNTL,44.47147,-112.98191,NaN,0.000,0.584,16.299999,2996,21.693270,...,15.414333,0.000,1.021333,15.414333,0.000,1997,8.441999,0.000000,24.653500,3
3,1996-10-04,318:MT:SNTL,44.47147,-112.98191,NaN,0.000,2.769,14.864000,2996,21.693270,...,14.920333,0.000,1.458250,15.276750,0.000,1997,8.816500,0.000000,33.470001,4
4,1996-10-05,318:MT:SNTL,44.47147,-112.98191,NaN,0.000,2.755,16.297001,2996,21.693270,...,15.820333,0.000,1.717600,15.480800,0.000,1997,9.526001,0.000000,42.996002,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146545,2020-05-28,924:MT:SNTL,44.65866,-111.09199,0.0,0.000,1.235,21.813000,2035,1.305896,...,18.629333,0.765,0.284714,12.022286,33.836,2020,11.524000,360.126007,-942.330505,241
146546,2020-05-29,924:MT:SNTL,44.65866,-111.09199,0.0,0.000,0.026,24.500999,2035,1.305896,...,21.185666,0.765,0.276143,14.380286,18.482,2020,12.263499,360.126007,-930.067017,242
146547,2020-05-30,924:MT:SNTL,44.65866,-111.09199,0.0,0.726,1.743,27.243999,2035,1.305896,...,24.519333,0.726,0.532857,17.315143,12.930,2020,14.493500,360.851990,-915.573486,243
146548,2020-05-31,924:MT:SNTL,44.65866,-111.09199,0.0,0.480,3.636,28.344999,2035,1.305896,...,26.696666,1.206,1.157286,20.901143,2.620,2020,15.990499,361.332001,-899.583008,244


## Step 6: Train/Test Splitting

This step will perform train/test splitting for use in the machine learning model. Because the data is so temporal in nature, the train/test split will be performed by removing the last 5 years of the dataset (20% of the data). 

In [25]:
# first, drop all remaining NaNs
# these were left from the NaN step so the feature calculation worked
stations_ml_df = stations_ml_df.dropna(subset=['swe_mm']).copy()

In [26]:
# check that there are no NaNs in the ml df

# Count total missing values across the entire DataFrame
total_nans = stations_ml_df.isna().sum().sum()

# Check NaNs
if total_nans > 0:
    print(f'DataFrame is not ready for train/test split. Found {total_nans} NaNs.')
else:
    print('DataFrame is ready for train/test split and input into machine learning model!')

DataFrame is ready for train/test split and input into machine learning model!


In [27]:
# split the dataset
stations_train_df = stations_ml_df[stations_ml_df['water_year'] < 2016].copy()
stations_test_df = stations_ml_df[stations_ml_df['water_year'] >= 2016].copy()

In [28]:
# get max train and min test year
max_train_year = stations_train_df['water_year'].max()
min_test_year = stations_test_df['water_year'].min()

# check that the split worked
if max_train_year >= min_test_year:
    print(f'Train/Test split unsuccessful. Train and Test datasets share year: {max_train_year}')
else:
    print('Train/Test split successful! Train and Test datasets do not overlap')

Train/Test split successful! Train and Test datasets do not overlap


## Step 7: Apply Feature Engineering to full PRISM dataset

In [7]:
slice = prism_ds['tmin'].sel(time = '2000-10-01').squeeze()

slice.hvplot(
    x = 'lon',
    y = 'lat',
    geo = True
)

:Image   [lon,lat]   (GDAL Band Number 1)

In [8]:
# define a couple of functions to perform feature engineering

def roll_mean_da(da, num_days):
    '''
    Calculate a rolling mean on an Xarray.DataArray.

    Args:
    -----
    da (xarray.DataArray):
        DataArray of temperature data timeseries
    num_days (int):
        Number of days for rolling window

    Returns:
    --------
    rolling_mean_da (xarray.DataArray):
        DataArray of rolling mean window
    '''

    rolling_mean_da = da.rolling(time=num_days, min_periods=1).mean()

    return rolling_mean_da

def roll_sum_da(da, num_days):
    '''
    Calculate a rolling sum on an Xarray.DataArray.

    Args:
    -----
    da (xarray.DataArray):
        DataArray of precipitation data timeseries
    num_days (int):
        Number of days for rolling window

    Returns:
    --------
    rolling_sum_da (xarray.DataArray):
        DataArray of rolling mean window
    '''

    rolling_sum_da = da.rolling(time=num_days, min_periods=1).sum()

    return rolling_sum_da

In [9]:
prism_ds

<xarray.Dataset> Size: 372MB
Dimensions:  (time: 7328, lat: 51, lon: 83)
Coordinates:
  * lon      (lon) float64 664B -113.9 -113.9 -113.8 ... -110.6 -110.5 -110.5
  * lat      (lat) float64 408B 46.46 46.42 46.38 46.33 ... 44.46 44.42 44.38
  * time     (time) datetime64[ns] 59kB 1990-10-01 1990-10-02 ... 2020-06-01
    crs      int64 8B 0
Data variables:
    ppt      (time, lat, lon) float32 124MB ...
    tmin     (time, lat, lon) float32 124MB ...
    tmax     (time, lat, lon) float32 124MB ...
Attributes:
    Conventions:  CF-1.5
    GDAL:         GDAL 3.12.0 "Chicoutimi", released 2025/11/03
    history:      Thu Jun 18 17:05:52 2026: GDAL CreateCopy( /nfs/pancake/u5/...

In [10]:
# water year
water_year_values = prism_ds['time'].dt.year + (prism_ds['time'].dt.month >= 10).astype(int)
prism_ds = prism_ds.assign_coords(water_year = water_year_values)

# day of water year
time_index = prism_ds.time.to_index()
wy_start_date = pd.to_datetime(
    # get water year, subtract 1 (since the water year started the previous year)
    (prism_ds['water_year'] - 1)
    # set to a string and add the month and day start
    .astype(str) + '-10-01'
    )
dowy_values = (prism_ds['time'] - wy_start_date).dt.days + 1
prism_ds = prism_ds.assign_coords(dowy = dowy_values)

# rolling 3 day mean of tmin
prism_ds['prism_tmin_c_roll_3d'] = prism_ds['tmin'].groupby('water_year').map(roll_mean_da, num_days=3)

# rolling 3 day mean of tmax
prism_ds['prism_tmax_c_roll_3d'] = prism_ds['tmax'].groupby('water_year').map(roll_mean_da, num_days=3)

# rolling 3 day total of precip
prism_ds['prism_ppt_mm_roll_3d'] = prism_ds['ppt'].groupby('water_year').map(roll_sum_da, num_days=3)

# rolling 7 day windows
prism_ds['prism_tmin_c_roll_7d'] = prism_ds['tmin'].groupby('water_year').map(roll_mean_da, num_days=7)
prism_ds['prism_tmax_c_roll_7d'] = prism_ds['tmax'].groupby('water_year').map(roll_mean_da, num_days=7)
prism_ds['prism_ppt_mm_roll_7d'] = prism_ds['ppt'].groupby('water_year').map(roll_sum_da, num_days=7)

# cumulative precip
prism_ds['wy_cumul_precip_mm'] = prism_ds['ppt'].groupby('water_year').cumsum(dim='time')

# tmean
prism_ds['prism_tmean_c'] = (prism_ds['tmin'] + prism_ds['tmax'])/2

# cumulative degree > 0 days
positive_tmean_days = prism_ds['prism_tmean_c'].clip(min=0)
prism_ds['wy_cumul_pos_degree_days'] = positive_tmean_days.groupby('water_year').cumsum(dim='time')

In [14]:
# merge dem data in ds

# rename dem coords and drop dem band
dem_rn_ds = dem_dataset.rename({'x': 'lon', 'y': 'lat'}).squeeze()

# ensure dem is aligned to PRISM grid
dem_aligned_ds = dem_rn_ds.reindex_like(prism_ds, method = 'nearest')

# merge dem with prism_ds
#prism_dem_ds = xr.merge([prism_ds, dem_aligned_ds], join = 'exact')

In [18]:
# flatten dem_ds
dem_df = dem_aligned_ds.to_dataframe()

In [ ]:
# convert prism_ds to df

# get length of time steps in ds
time_steps = prism_ds.time.values

# set chunks of timesteps to process
chunk_size = 500  # Process 500 days at a time

# initialize list
prism_dem_chunks = []

print(f"Starting Dataset to DataFrame conversion across {len(time_steps)} time steps...")

# loop through chunks and convert to df
for i in range(0, len(time_steps), chunk_size):
    # Grab the specific block of dates
    time_chunk = time_steps[i : i + chunk_size]
    print(f"Processing chunk {i} to {i + len(time_chunk)}.")

    # Slice the climate dataset for just this time window
    prism_chunk = prism_ds.sel(time=time_chunk)

    # Convert just this small chunk to a dataframe (HDF reader handles this easily)
    climate_chunk_df = prism_chunk.to_dataframe()

    # Join the static terrain features onto this time chunk
    # Pandas automatically matches the 'lat' and 'lon' indices
    prism_dem_chunk = climate_chunk_df.join(dem_df, how="left").dropna()

    # append
    prism_dem_chunks.append(prism_dem_chunk)

# concatenate the chunks
prism_dem_df = pd.concat(prism_dem_chunks)

Starting inference across 7328 time steps...
Processing chunk 0 to 500...
Processing chunk 500 to 1000...
Processing chunk 1000 to 1500...
Processing chunk 1500 to 2000...
Processing chunk 2000 to 2500...
Processing chunk 2500 to 3000...
Processing chunk 3000 to 3500...
Processing chunk 3500 to 4000...
Processing chunk 4000 to 4500...
Processing chunk 4500 to 5000...
Processing chunk 5000 to 5500...
Processing chunk 5500 to 6000...
Processing chunk 6000 to 6500...
Processing chunk 6500 to 7000...
Processing chunk 7000 to 7328...


In [ ]:
# check the df
print(prism_dem_df.columns)
prism_dem_df.head()

## need to drop crs, band, spatial ref

Index(['ppt', 'tmin', 'tmax', 'crs', 'water_year', 'dowy',
       'prism_tmin_c_roll_3d', 'prism_tmax_c_roll_3d', 'prism_ppt_mm_roll_3d',
       'prism_tmin_c_roll_7d', 'prism_tmax_c_roll_7d', 'prism_ppt_mm_roll_7d',
       'wy_cumul_precip_mm', 'prism_tmean_c', 'wy_cumul_pos_degree_days',
       'band', 'spatial_ref', 'elevation', 'slope', 'aspect', 'aspect_north',
       'aspect_east'],
      dtype='object')


In [23]:
# drop unnecessary columns

drop_cols = ['band', 'spatial_ref', 'crs']
prism_dem_df = prism_dem_df.drop(columns = drop_cols, errors = 'ignore')

In [25]:
# recheck the df
print(prism_dem_df.columns)
prism_dem_df.head()

Index(['ppt', 'tmin', 'tmax', 'water_year', 'dowy', 'prism_tmin_c_roll_3d',
       'prism_tmax_c_roll_3d', 'prism_ppt_mm_roll_3d', 'prism_tmin_c_roll_7d',
       'prism_tmax_c_roll_7d', 'prism_ppt_mm_roll_7d', 'wy_cumul_precip_mm',
       'prism_tmean_c', 'wy_cumul_pos_degree_days', 'elevation', 'slope',
       'aspect', 'aspect_north', 'aspect_east'],
      dtype='object')


ppt   tmin       tmax  water_year  dowy  \
time       lat       lon                                                    
1990-10-01 46.458337 -112.291671  0.0  2.794  17.695999        1991     1   
                     -112.250004  0.0  3.345  18.033001        1991     1   
                     -112.208338  0.0  2.534  17.188999        1991     1   
           46.416670 -112.291671  0.0  1.844  16.056999        1991     1   
                     -112.250004  0.0  1.648  15.697000        1991     1   

                                  prism_tmin_c_roll_3d  prism_tmax_c_roll_3d  \
time       lat       lon                                                       
1990-10-01 46.458337 -112.291671                 2.794             17.695999   
                     -112.250004                 3.345             18.033001   
                     -112.208338                 2.534             17.188999   
           46.416670 -112.291671                 1.844             16.056999   
                     -112.250004                 1.648             15.697000   

                                  prism_ppt_mm_roll_3d  prism_tmin_c_roll_7d  \
time       lat       lon                                                       
1990-10-01 46.458337 -112.291671                   0.0                 2.794   
                     -112.250004                   0.0                 3.345   
                     -112.208338                   0.0                 2.534   
           46.416670 -112.291671                   0.0                 1.844   
                     -112.250004                   0.0                 1.648   

                                  prism_tmax_c_roll_7d  prism_ppt_mm_roll_7d  \
time       lat       lon                                                       
1990-10-01 46.458337 -112.291671             17.695999                   0.0   
                     -112.250004             18.033001                   0.0   
                     -112.208338             17.188999                   0.0   
           46.416670 -112.291671             16.056999                   0.0   
                     -112.250004             15.697000                   0.0   

                                  wy_cumul_precip_mm  prism_tmean_c  \
time       lat       lon                                              
1990-10-01 46.458337 -112.291671                 0.0        10.2450   
                     -112.250004                 0.0        10.6890   
                     -112.208338                 0.0         9.8615   
           46.416670 -112.291671                 0.0         8.9505   
                     -112.250004                 0.0         8.6725   

                                  wy_cumul_pos_degree_days  elevation  \
time       lat       lon                                                
1990-10-01 46.458337 -112.291671                   10.2450       2061   
                     -112.250004                   10.6890       2040   
                     -112.208338                    9.8615       2181   
           46.416670 -112.291671                    8.9505       2245   
                     -112.250004                    8.6725       2292   

                                      slope      aspect  aspect_north  \
time       lat       lon                                                
1990-10-01 46.458337 -112.291671  11.218369  358.451843     -0.027017   
                     -112.250004  15.176164  102.236107      0.977283   
                     -112.208338  15.415931  108.669273      0.947382   
           46.416670 -112.291671   8.808415  220.179230     -0.645181   
                     -112.250004  10.946354  213.690063     -0.554700   

                                  aspect_east  
time       lat       lon                       
1990-10-01 46.458337 -112.291671     0.999635  
                     -112.250004    -0.211941  
                     -112.208338    -0.320105  
           46.416670 -112.291671    -0.764030  
 

In [ ]:
# rename cols
prism_dem_df.rename(columns = {
    'ppt': 'prism_ppt_mm', 
    'tmin': 'prism_tmin_c',
    'tmax': 'prism_tmax_c',
    'wy_cumul_pos_degree_days': 'wy_cumul_melt_days'},
    inplace = True)

prism_ppt_mm  prism_tmin_c  prism_tmax_c  \
time       lat       lon                                                     
1990-10-01 46.458337 -112.291671           0.0         2.794     17.695999   
                     -112.250004           0.0         3.345     18.033001   
                     -112.208338           0.0         2.534     17.188999   
           46.416670 -112.291671           0.0         1.844     16.056999   
                     -112.250004           0.0         1.648     15.697000   

                                  water_year  dowy  prism_tmin_c_roll_3d  \
time       lat       lon                                                   
1990-10-01 46.458337 -112.291671        1991     1                 2.794   
                     -112.250004        1991     1                 3.345   
                     -112.208338        1991     1                 2.534   
           46.416670 -112.291671        1991     1                 1.844   
                     -112.250004        1991     1                 1.648   

                                  prism_tmax_c_roll_3d  prism_ppt_mm_roll_3d  \
time       lat       lon                                                       
1990-10-01 46.458337 -112.291671             17.695999                   0.0   
                     -112.250004             18.033001                   0.0   
                     -112.208338             17.188999                   0.0   
           46.416670 -112.291671             16.056999                   0.0   
                     -112.250004             15.697000                   0.0   

                                  prism_tmin_c_roll_7d  prism_tmax_c_roll_7d  \
time       lat       lon                                                       
1990-10-01 46.458337 -112.291671                 2.794             17.695999   
                     -112.250004                 3.345             18.033001   
                     -112.208338                 2.534             17.188999   
           46.416670 -112.291671                 1.844             16.056999   
                     -112.250004                 1.648             15.697000   

                                  prism_ppt_mm_roll_7d  wy_cumul_precip_mm  \
time       lat       lon                                                     
1990-10-01 46.458337 -112.291671                   0.0                 0.0   
                     -112.250004                   0.0                 0.0   
                     -112.208338                   0.0                 0.0   
           46.416670 -112.291671                   0.0                 0.0   
                     -112.250004                   0.0                 0.0   

                                  prism_tmean_c  wy_cumul_melt_days  \
time       lat       lon                                              
1990-10-01 46.458337 -112.291671        10.2450             10.2450   
                     -112.250004        10.6890             10.6890   
                     -112.208338         9.8615              9.8615   
           46.416670 -112.291671         8.9505              8.9505   
                     -112.250004         8.6725              8.6725   

                                  elevation      slope      aspect  \
time       lat       lon                                             
1990-10-01 46.458337 -112.291671       2061  11.218369  358.451843   
                     -112.250004       2040  15.176164  102.236107   
                     -112.208338       2181  15.415931  108.669273   
           46.416670 -112.291671       2245   8.808415  220.179230   
                     -112.250004       2292  10.946354  213.690063   

                                  aspect_north  aspect_east  
time       lat       lon                                     
1990-10-01 46.458337 -112.291671     -0.027017     0.999635  
                     -112.250004      0.977283    -0.211941  
                     -112.208338      0.947382  

In [31]:
# reset index
prism_dem_df['date'] = prism_dem_df.index.get_level_values('time')
prism_dem_df['latitude'] = prism_dem_df.index.get_level_values('lat')
prism_dem_df['longitude'] = prism_dem_df.index.get_level_values('lon')

# check the df
prism_dem_df.head()

prism_ppt_mm  prism_tmin_c  prism_tmax_c  \
time       lat       lon                                                     
1990-10-01 46.458337 -112.291671           0.0         2.794     17.695999   
                     -112.250004           0.0         3.345     18.033001   
                     -112.208338           0.0         2.534     17.188999   
           46.416670 -112.291671           0.0         1.844     16.056999   
                     -112.250004           0.0         1.648     15.697000   

                                  water_year  dowy  prism_tmin_c_roll_3d  \
time       lat       lon                                                   
1990-10-01 46.458337 -112.291671        1991     1                 2.794   
                     -112.250004        1991     1                 3.345   
                     -112.208338        1991     1                 2.534   
           46.416670 -112.291671        1991     1                 1.844   
                     -112.250004        1991     1                 1.648   

                                  prism_tmax_c_roll_3d  prism_ppt_mm_roll_3d  \
time       lat       lon                                                       
1990-10-01 46.458337 -112.291671             17.695999                   0.0   
                     -112.250004             18.033001                   0.0   
                     -112.208338             17.188999                   0.0   
           46.416670 -112.291671             16.056999                   0.0   
                     -112.250004             15.697000                   0.0   

                                  prism_tmin_c_roll_7d  prism_tmax_c_roll_7d  \
time       lat       lon                                                       
1990-10-01 46.458337 -112.291671                 2.794             17.695999   
                     -112.250004                 3.345             18.033001   
                     -112.208338                 2.534             17.188999   
           46.416670 -112.291671                 1.844             16.056999   
                     -112.250004                 1.648             15.697000   

                                  ...  prism_tmean_c  wy_cumul_melt_days  \
time       lat       lon          ...                                      
1990-10-01 46.458337 -112.291671  ...        10.2450             10.2450   
                     -112.250004  ...        10.6890             10.6890   
                     -112.208338  ...         9.8615              9.8615   
           46.416670 -112.291671  ...         8.9505              8.9505   
                     -112.250004  ...         8.6725              8.6725   

                                  elevation      slope      aspect  \
time       lat       lon                                             
1990-10-01 46.458337 -112.291671       2061  11.218369  358.451843   
                     -112.250004       2040  15.176164  102.236107   
                     -112.208338       2181  15.415931  108.669273   
           46.416670 -112.291671       2245   8.808415  220.179230   
                     -112.250004       2292  10.946354  213.690063   

                                  aspect_north  aspect_east       date  \
time       lat       lon                                                 
1990-10-01 46.458337 -112.291671     -0.027017     0.999635 1990-10-01   
                     -112.250004      0.977283    -0.211941 1990-10-01   
                     -112.208338      0.947382    -0.320105 1990-10-01   
           46.416670 -112.291671     -0.645181    -0.764030 1990-10-01   
                     -112.250004     -0.554700    -0.832050 1990-10-01   

                                   latitude   longitude  
time       lat       lon                                 
1990-10-01 46.458337 -112.291671  46.458337 -112.291671  
                     -112.250004  46.458337 -112.250004  
                     -112.208338  46.458337 -112.208338  

In [ ]:
# reorder the df to match the ML data

# model features order
model_features_in_order = [
    # date
    'date',
    # lat and lon
    'latitude', 'longitude', 
    # raw prism data
    'prism_ppt_mm', 'prism_tmin_c', 'prism_tmax_c', 
    # topographic data
    'elevation', 'slope', 'aspect_north', 'aspect_east', 
    # lagged prism variables 7-day
    'prism_tmin_c_roll_7d', 'prism_tmax_c_roll_7d', 'prism_ppt_mm_roll_7d',
    # cumulative precip and positive degree days
    'wy_cumul_precip_mm', 'wy_cumul_melt_days',
    # day of water year
    'dowy']

# reorder
prism_dem_reord_df = prism_dem_df[model_features_in_order]

# check
prism_dem_reord_df.head()

In [45]:
# drop NaNs
prism_dem_clean_df = prism_dem_reord_df.dropna()

In [46]:
# check it out
print(prism_dem_clean_df.isna().sum())

date                    0
latitude                0
longitude               0
prism_ppt_mm            0
prism_tmin_c            0
prism_tmax_c            0
elevation               0
slope                   0
aspect_north            0
aspect_east             0
prism_tmin_c_roll_7d    0
prism_tmax_c_roll_7d    0
prism_ppt_mm_roll_7d    0
wy_cumul_precip_mm      0
wy_cumul_melt_days      0
dowy                    0
dtype: int64


PRISM dataset is clean!

## Step 8: Save datasets for use in the model

In [47]:
# set save dir
ml_save_dir = Path(cleaned_data_dir, 'machine_learning')
os.makedirs(ml_save_dir, exist_ok=True)

# set save paths
master_station_path = Path(ml_save_dir, 'stations_master.parquet')
train_station_path = Path(ml_save_dir, 'stations_train.parquet')
test_station_path = Path(ml_save_dir, 'stations_test.parquet')
prism_raster_path = Path(ml_save_dir, 'prism_raster.parquet')

In [ ]:
# save dfs

stations_ml_df.to_parquet(master_station_path, index=False)
stations_train_df.to_parquet(train_station_path, index=False)
stations_test_df.to_parquet(test_station_path, index=False)
prism_dem_clean_df.to_parquet(prism_raster_path, index=False)